# Movie recommendation system

**Purpose**

This notebook builds a content-based movie recommendation system using a dataset of ~4,800 films. Given a movie title, each system returns the 10 most similar titles based on text and/or numerical features.

**Technologies**
- Python, Jupyter Notebook
- pandas and numpy for data handling
- scikit-learn for vectorization and similarity metrics (CountVectorizer, cosine similarity, Euclidean distance, Manhattan distance)
- nltk for basic NLP preprocessing (stemming)

**Data techniques**
- Feature engineering from movie metadata (cast, crew, keywords, genres, overview)
- Text normalization and stemming
- Porter stemming
- Bag-of-words vectorization with CountVectorizer
- Feature concatenation with normalized numerical signals
- Similarity/distance computation and top-N retrieval

**Author:** Gui Freire Oliveira

In [1]:
# Install necessary packages if needed
# !pip install -q -U watermark
# !pip install numpy
# !pip install sklearn
# !pip install nltk
# !pip install ast
# !pip install pandas

In [2]:
# Imports for data handling, NLP, and similarity computations
import ast
import nltk
import sklearn
import numpy as np
import pandas as pd
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics.pairwise import manhattan_distances
pd.options.mode.chained_assignment = None

In [3]:
# Package versions used in this project
%reload_ext watermark
%watermark -a "Gui Freire Oliveira" 

Author: Gui Freire Oliveira



# I - Loading and understanding data

In [4]:
# Load the movies dataset
df_movies = pd.read_csv("data/dataset_movies.csv")

In [5]:
# Quick check of the dataframe shape
df_movies.shape

(4803, 20)

In [6]:
# Preview a few rows for a sanity check
df_movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [7]:
# Load the movies dataset
df_cast = pd.read_csv("data/dataset_cast.csv")

In [8]:
# Quick check of the dataframe shape
df_cast.shape

(4803, 4)

In [9]:
# Preview a few rows for a sanity check
df_cast.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


# II - Organization of Text Data

In [10]:
# merging the dataframes on 'title' column
df_movies = df_movies.merge(df_cast, on = 'title')

In [11]:
# Check the current dataframe shape
df_movies.shape

(4809, 23)

In [12]:
# Sample a few rows of the merged dataset to inspect the columns
df_movies.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...",...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,49026,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]",...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,49529,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [13]:
# Retrieve basic info about the dataframe (types and nulls)
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [14]:
# let us filter the dataframe to gather only the relevant information for the recommendation system
df_movies = df_movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'popularity']]
# maybe popularity is not that relevant, because it will assume people watch movies based on how popular it is
# but I still wanna see the effect of including it!

In [15]:
df_movies.head(10)

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,150.437577
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,139.082615
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3,107.376788
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",7.6,112.312950
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",6.1,43.926995
5,559,Spider-Man 3,The seemingly invincible Spider-Man goes up ag...,"[{""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""na...","[{""id"": 851, ""name"": ""dual identity""}, {""id"": ...","[{""cast_id"": 30, ""character"": ""Peter Parker / ...","[{""credit_id"": ""52fe4252c3a36847f80151a5"", ""de...",5.9,115.699814
6,38757,Tangled,When the kingdom's most wanted-and most charmi...,"[{""id"": 16, ""name"": ""Animation""}, {""id"": 10751...","[{""id"": 1562, ""name"": ""hostage""}, {""id"": 2343,...","[{""cast_id"": 34, ""character"": ""Flynn Rider (vo...","[{""credit_id"": ""52fe46db9251416c91062101"", ""de...",7.4,48.681969
7,99861,Avengers: Age of Ultron,When Tony Stark tries to jumpstart a dormant p...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 8828, ""name"": ""marvel comic""}, {""id"": ...","[{""cast_id"": 76, ""character"": ""Tony Stark / Ir...","[{""credit_id"": ""55d5f7d4c3a3683e7e0016eb"", ""de...",7.3,134.279229
8,767,Harry Potter and the Half-Blood Prince,"As Harry begins his sixth year at Hogwarts, he...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 616, ""name"": ""witch""}, {""id"": 2343, ""n...","[{""cast_id"": 3, ""character"": ""Harry Potter"", ""...","[{""credit_id"": ""52fe4273c3a36847f801fab1"", ""de...",7.4,98.885637
9,209112,Batman v Superman: Dawn of Justice,Fearing the actions of a god-like Super Hero l...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 7002...","[{""cast_id"": 18, ""character"": ""Bruce Wayne / B...","[{""credit_id"": ""553bf23692514135c8002886"", ""de...",5.7,155.790452


# III - Exploratory Data Analysis and Data Cleaning

In [16]:
# Confirm the shape after filtering/merging
df_movies.shape

(4809, 9)

In [17]:
# check absent values and their lines
df_movies.isnull().sum()

movie_id        0
title           0
overview        3
genres          0
keywords        0
cast            0
crew            0
vote_average    0
popularity      0
dtype: int64

In [18]:
# Display the rows with missing values
df_movies[df_movies.isnull().any(axis=1)]

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
2658,370980,Chiamatemi Francesco - Il Papa della gente,NaN,"[{""id"": 18, ""name"": ""Drama""}]","[{""id"": 717, ""name"": ""pope""}, {""id"": 5565, ""na...","[{""cast_id"": 5, ""character"": ""Jorge Mario Berg...","[{""credit_id"": ""5660019ac3a36875f100252b"", ""de...",7.3,0.738646
4145,459488,"To Be Frank, Sinatra at 100",NaN,"[{""id"": 99, ""name"": ""Documentary""}]","[{""id"": 6027, ""name"": ""music""}, {""id"": 225822,...","[{""cast_id"": 0, ""character"": ""Narrator"", ""cred...","[{""credit_id"": ""592b25e4c3a368783e065a2f"", ""de...",0.0,0.050625
4437,292539,Food Chains,NaN,"[{""id"": 99, ""name"": ""Documentary""}]",[],[],"[{""credit_id"": ""5470c3b1c3a368085e000abd"", ""de...",7.4,0.795698


In [19]:
# Remove lines with absent values
df_movies.dropna(inplace = True)
df_movies.isnull().sum()

movie_id        0
title           0
overview        0
genres          0
keywords        0
cast            0
crew            0
vote_average    0
popularity      0
dtype: int64

In [20]:
# Check for duplicated rows/titles
df_movies.duplicated().sum()

0

In [21]:
# Re-check the shape after cleaning steps
df_movies.shape

(4806, 9)

# IV - Text processing using Abstract Syntax Trees

In [22]:
# Inspect the updated dataframe before AST processing
df_movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,150.437577
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,139.082615
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3,107.376788
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",7.6,112.312950
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",6.1,43.926995


In [23]:
# Some fields are a list of dictionaries. For example, genres are a list of genres of the movie, with keys and values
# I need to extract the data into a proper Python list. We will use ast.literal_eval for this
# Small example to understand the structure
ast.literal_eval('[{"id": 28, "name": "Action"}, \
                   {"id": 12, "name": "Adventure"}, \
                   {"id": 14, "name": "Fantasy"}, \
                   {"id": 878, "name": "Science Fiction"}]')

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [24]:
# Let us define a converter function, that will pick only the 'name' key in the list of dictionaries object.
# We don't need the ID in this case
def converter(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

In [25]:
# Test in the list above/below
test = converter('[{"id": 28, "name": "Action"}, \
         {"id": 12, "name": "Adventure"}, \
         {"id": 14, "name": "Fantasy"}, \
         {"id": 878, "name": "Science Fiction"}]')
print(test)

['Action', 'Adventure', 'Fantasy', 'Science Fiction']


In [26]:
# Let us apply the converter to genres and keywords columns
columns = ['genres', 'keywords']
for column in columns:
    df_movies[column] = df_movies[column].apply(converter)

In [27]:
# Validate the intermediate result
df_movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,150.437577
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,139.082615
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3,107.376788
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",7.6,112.312950
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",6.1,43.926995


The same happens for cast and crew, but now we don't need all the names strategy: 
- for cast, we will retrieve only the first three names (main actresses/actors);
- for crew, we will retrieve only the first name, the director. These are the most important data in these columns. Let us check the data structure of these columns 

In [28]:
df_movies["cast"].iloc[1][:500]
# There is no way to find out (without looking at the dataframe source)
# if the cast is in importance order. But that is a reasonable assumption.
# We see in some lines that it starts with the main actresses/actors

'[{"cast_id": 4, "character": "Captain Jack Sparrow", "credit_id": "52fe4232c3a36847f800b50d", "gender": 2, "id": 85, "name": "Johnny Depp", "order": 0}, {"cast_id": 5, "character": "Will Turner", "credit_id": "52fe4232c3a36847f800b511", "gender": 2, "id": 114, "name": "Orlando Bloom", "order": 1}, {"cast_id": 6, "character": "Elizabeth Swann", "credit_id": "52fe4232c3a36847f800b515", "gender": 1, "id": 116, "name": "Keira Knightley", "order": 2}, {"cast_id": 12, "character": "William \\"Bootstrap'

In [29]:
df_movies["crew"].iloc[0][:500]
# There is a key 'job', where we can find the director. We break the loop to take only the first director (if >1)

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0,'

In [30]:
# Modified converter helpers:
# - converter3 keeps up to the first 3 names (useful for cast)
# - converter1 extracts the director name (useful for crew)
def converter3(obj):

    # Collect up to 3 names from the parsed list of dicts.
    L = []
    counter = 0

    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break

    return L

def converter1(obj):

    # Extract the director name from the crew list (first match only).
    L = []

    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break

    return L


In [31]:
# Apply the helper functions to parse names/crew
df_movies['cast'] = df_movies['cast'].apply(converter3)
df_movies['crew'] = df_movies['crew'].apply(converter1)

In [32]:
# Validate the intermediate result
df_movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],7.2,150.437577
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],6.9,139.082615
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],6.3,107.376788
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],7.6,112.312950
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],6.1,43.926995


# V - Cleaning of Text Data

In [33]:
# In the overview column, we will break the information to a list, splitting by space
df_movies['overview'] = df_movies['overview'].apply(lambda x:x.split())
df_movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],7.2,150.437577
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],6.9,139.082615
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],6.3,107.376788
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],7.6,112.312950
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],6.1,43.926995


In [34]:
# Now, remove internal spaces in multi-word entries so they're treated as single tokens during vectorization
columns = ['genres', 'keywords', 'cast', 'crew']
for column in columns:
    df_movies[column] = df_movies[column].apply(lambda x:[i.replace(" ","") for i in x])
df_movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],7.2,150.437577
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],6.9,139.082615
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],6.3,107.376788
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],7.6,112.312950
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],6.1,43.926995


# VI - Normalize numerical data

In [35]:
# Let us normalize our numerical data
# Vote_average should be normalized by 10, since it is the maximum vote possible
df_movies['vote_average'] = df_movies['vote_average'].apply(lambda x: x/10.0)

In [36]:
# I will normalize popularity by its maximum value
max_popularity = np.max(df_movies[['popularity']])
# Let us first check which movie has the maximum popularity among viewers
df_movies[df_movies['popularity'] == max_popularity]

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
546,211672,Minions,"[Minions, Stuart,, Kevin, and, Bob, are, recru...","[Family, Animation, Adventure, Comedy]","[assistant, aftercreditsstinger, duringcredits...","[SandraBullock, JonHamm, MichaelKeaton]",[KyleBalda],0.64,875.581305


In [37]:
# Normalize the text to keep a consistent format
df_movies['popularity'] = df_movies['popularity'].apply(lambda x: x/max_popularity)

In [38]:
# Verify the effect of the transformation
df_movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],0.72,0.171815
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],0.69,0.158846
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],0.63,0.122635
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],0.76,0.128272
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],0.61,0.050169


# VII - Preparing to vectorize text data

In [39]:
# Concatenate all text features into a single 'tags' column; individual text columns will be dropped afterwards
# to reduce dimensionality
df_movies['tags'] = df_movies['overview'] + \
                        df_movies['genres'] + \
                        df_movies['keywords'] + \
                        df_movies['cast'] + \
                        df_movies['crew']

In [40]:
# Verify the effect of the transformation
df_movies['tags'].iloc[0][:10]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to']

In [41]:
# Check the dataframe after the transformation
df_movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,popularity,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, ScienceFiction]","[cultureclash, future, spacewar, spacecolony, ...","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron],0.72,0.171815,"[In, the, 22nd, century,, a, paraplegic, Marin..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...","[Adventure, Fantasy, Action]","[ocean, drugabuse, exoticisland, eastindiatrad...","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski],0.69,0.158846,"[Captain, Barbossa,, long, believed, to, be, d..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...","[Action, Adventure, Crime]","[spy, basedonnovel, secretagent, sequel, mi6, ...","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes],0.63,0.122635,"[A, cryptic, message, from, Bond’s, past, send..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...","[Action, Crime, Drama, Thriller]","[dccomics, crimefighter, terrorist, secretiden...","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan],0.76,0.128272,"[Following, the, death, of, District, Attorney..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...","[Action, Adventure, ScienceFiction]","[basedonnovel, mars, medallion, spacetravel, p...","[TaylorKitsch, LynnCollins, SamanthaMorton]",[AndrewStanton],0.61,0.050169,"[John, Carter, is, a, war-weary,, former, mili..."


In [42]:
# Build final dataset. We remove any other text column apart from tags
df_movies_new = df_movies[['movie_id', 'title', 'tags', 'vote_average', 'popularity']]
df_movies_new.head()

,movie_id,title,tags,vote_average,popularity
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...",0.72,0.171815
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...",0.69,0.158846
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...",0.63,0.122635
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...",0.76,0.128272
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...",0.61,0.050169


In [43]:
# Let us process just a bit further
# Join the list in a single string variable using space
df_movies_new['tags'] = df_movies_new['tags'].apply(lambda x:" ".join(x))
# Make all tokens lowercase for consistency
df_movies_new['tags'] = df_movies_new['tags'].apply(lambda x:x.lower())

In [44]:
# Quick check of the output
df_movies_new.head(10)

,movie_id,title,tags,vote_average,popularity
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...",0.72,0.171815
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...",0.69,0.158846
2,206647,Spectre,a cryptic message from bond’s past sends him o...,0.63,0.122635
3,49026,The Dark Knight Rises,following the death of district attorney harve...,0.76,0.128272
4,49529,John Carter,"john carter is a war-weary, former military ca...",0.61,0.050169
5,559,Spider-Man 3,the seemingly invincible spider-man goes up ag...,0.59,0.132141
6,38757,Tangled,when the kingdom's most wanted-and most charmi...,0.74,0.055600
7,99861,Avengers: Age of Ultron,when tony stark tries to jumpstart a dormant p...,0.73,0.153360
8,767,Harry Potter and the Half-Blood Prince,"as harry begins his sixth year at hogwarts, he...",0.74,0.112937
9,209112,Batman v Superman: Dawn of Justice,fearing the actions of a god-like super hero l...,0.57,0.177928


# VIII - Parse and vectorization

In [45]:
# Initialize the Porter Stemmer — it reduces each word to its base form (e.g. 'running' → 'run')
parser_ps = PorterStemmer()

In [46]:
# Basic stemming helper to normalize words before vectorization
def stem(text):

    # Collect stemmed tokens and join back into a single string.
    y = []

    for i in text.split():
        y.append(parser_ps.stem(i))

    return " ".join(y)


In [47]:
# Apply the stem to the 'tags' column. It will reduce it to lemmas and place the result in a single string
df_movies_new['tags'] = df_movies_new['tags'].apply(stem)

In [48]:
# Quick check of the stemming effect
df_movies_new.head(10)

,movie_id,title,tags,vote_average,popularity
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa...",0.72,0.171815
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c...",0.69,0.158846
2,206647,Spectre,a cryptic messag from bond’ past send him on a...,0.63,0.122635
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...,0.76,0.128272
4,49529,John Carter,"john carter is a war-weary, former militari ca...",0.61,0.050169
5,559,Spider-Man 3,the seemingli invinc spider-man goe up against...,0.59,0.132141
6,38757,Tangled,when the kingdom' most wanted-and most charmin...,0.74,0.055600
7,99861,Avengers: Age of Ultron,when toni stark tri to jumpstart a dormant pea...,0.73,0.153360
8,767,Harry Potter and the Half-Blood Prince,"as harri begin hi sixth year at hogwarts, he d...",0.74,0.112937
9,209112,Batman v Superman: Dawn of Justice,fear the action of a god-lik super hero left u...,0.57,0.177928


In [49]:
# now we create a vectorizer, removing stopwrods in english. 
# I'll fix the max_features to 5000, which will limit to the 5000 more frequent words
cv = CountVectorizer(max_features = 5000, stop_words = 'english')

Vectorization in natural language processing converts text into a numerical representation, typically a vector. This is necessary for machine learning algorithms, which require numerical inputs.

The `CountVectorizer` above builds a vocabulary from all unique words found across the documents and then counts how many times each word appears in each document.

`max_features = 5000` limits the vocabulary to the 5000 most frequent terms, and `stop_words = 'english'` removes common English words (e.g. "the", "is", "and") that carry little discriminative value.

In [50]:
# Create the vectors using the 'tags' column
vectors = cv.fit_transform(df_movies_new['tags']).toarray()

The `vectors` matrix produced by `fit_transform` represents the frequency of each stemmed word in each movie's tags string. Each entry means:

- `0`: the word does not appear in that movie's tags.
- `1`: the word appears once.
- `2` or more: the word appears multiple times — this can happen after stemming, since different surface forms (e.g. "run", "running", "runs") all reduce to the same stem and their counts accumulate.

This is expected because CountVectorizer builds a vocabulary of all unique words found in the input documents and then counts occurrences per document. Since we apply stemming with the `stem` function, different forms of the same word can be reduced to a single root, which can increase counts for that root in a given document (for example, "run", "running", and "runs" can collapse into the same base token).

In [51]:
# Check the shape. Each line represents a movie
print(type(vectors))
print(vectors.shape)

<class 'numpy.ndarray'>
(4806, 5000)


# IX - Building the Recommendation System

In [52]:
# first, let us include the popularity and the vote_average in the vectors we have constructed before
pop      = np.array(df_movies_new[['popularity']])
vote_avg = np.array(df_movies_new[['vote_average']])
# concatenate the arrays
vectorized_w_pop_n_vote = np.concatenate([vectors, pop], axis=1)
vectorized_w_pop_n_vote = np.concatenate([vectorized_w_pop_n_vote, vote_avg], axis=1)

In [53]:
# I will use the cosine similarity for this recommendation system               
similarity1 = cosine_similarity(vectorized_w_pop_n_vote)

In [54]:
# Recommendation system 1: text vectors + popularity + vote_average (cosine similarity)
def recommendation_system1(movie):

    # Get the index for the input movie title
    index = df_movies_new[df_movies_new['title'] == movie].index[0]

    # Sort by highest cosine similarity score
    scores = sorted(list(enumerate(similarity1[index])), reverse=True, key=lambda x: x[1])

    # Print the top 10 closest titles (skipping the movie itself)
    for i in scores[1:11]:
        print(df_movies_new.iloc[i[0]].title)


# X - Using the recommendation system

### What are the movie recommendations for someone who watched: Spectre using system 1?

In [55]:
recommendation_system1('Spectre')

Quantum of Solace
Skyfall
Never Say Never Again
From Russia with Love
Octopussy
Safe Haven
Diamonds Are Forever
Thunderball
Licence to Kill
Dr. No


### What are the movie recommendations for someone who watched: Minions using system 1?

In [56]:
recommendation_system1('Minions')

Despicable Me 2
Penguins of Madagascar
The Croods
Batman
Cars 2
Ice Age: Continental Drift
The Lego Movie
Despicable Me
Valiant
The Master of Disguise


### What are the movie recommendations for someone who watched: 300 using system 1?

In [57]:
recommendation_system1('300')

300: Rise of an Empire
The Thin Red Line
Red Cliff
Fort McCoy
Alexander
One Man's Hero
Underworld: Rise of the Lycans
We Were Soldiers
Highlander
Glory


# Study 1 - What if we remove 'popularity'?

In [58]:
# Build system 2 features: text vectors + vote_average (cosine similarity)
vectorized_w_vote = np.concatenate([vectors, vote_avg], axis=1)
similarity2 = cosine_similarity(vectorized_w_vote)

# Recommendation system 2: text vectors + vote_average (cosine similarity)
def recommendation_system2(movie):

    # Get the index for the input movie title
    index = df_movies_new[df_movies_new['title'] == movie].index[0]

    # Sort by highest cosine similarity score
    scores = sorted(list(enumerate(similarity2[index])), reverse=True, key=lambda x: x[1])

    # Print the top 10 closest titles (skipping the movie itself)
    for i in scores[1:11]:
        print(df_movies_new.iloc[i[0]].title)


### What are the movie recommendations for someone who watched: Spectre using system 2?

In [59]:
recommendation_system2('Spectre')

Quantum of Solace
Skyfall
Never Say Never Again
From Russia with Love
Octopussy
Safe Haven
Diamonds Are Forever
Thunderball
Licence to Kill
Dr. No


### What are the movie recommendations for someone who watched: Minions using system 2?

In [60]:
recommendation_system2('Minions')

Despicable Me 2
Penguins of Madagascar
The Croods
Batman
Cars 2
Ice Age: Continental Drift
The Lego Movie
Despicable Me
The Master of Disguise
Valiant


### What are the movie recommendations for someone who watched: 300 using system 2?

In [61]:
recommendation_system2('300')

300: Rise of an Empire
The Thin Red Line
Red Cliff
Fort McCoy
Alexander
One Man's Hero
Underworld: Rise of the Lycans
We Were Soldiers
Highlander
Glory


# Study 2 - What if we remove 'popularity' and 'vote_average'?

In [62]:
# Build system 3 features: text vectors only (cosine similarity)
# This is a bit redundant, but it keeps the same workflow as the other systems.
vectorized = vectors[:, :]
similarity3 = cosine_similarity(vectorized)

# Recommendation system 3: text vectors only (cosine similarity)
def recommendation_system3(movie):

    # Get the index for the input movie title
    index = df_movies_new[df_movies_new['title'] == movie].index[0]

    # Sort by highest cosine similarity score
    scores = sorted(list(enumerate(similarity3[index])), reverse=True, key=lambda x: x[1])

    # Print the top 10 closest titles (skipping the movie itself)
    for i in scores[1:11]:
        print(df_movies_new.iloc[i[0]].title)


### What are the movie recommendations for someone who watched: Spectre using system 3?

In [63]:
recommendation_system3('Spectre')

Quantum of Solace
Skyfall
Never Say Never Again
From Russia with Love
Octopussy
Diamonds Are Forever
Thunderball
Safe Haven
Licence to Kill
Dr. No


### What are the movie recommendations for someone who watched: Minions using system 3?

In [64]:
recommendation_system3('Minions')

Despicable Me 2
The Croods
Penguins of Madagascar
Batman
Cars 2
The Master of Disguise
Ice Age: Continental Drift
Valiant
Despicable Me
Free Birds


### What are the movie recommendations for someone who watched: 300 using system 3?

In [65]:
recommendation_system3('300')

300: Rise of an Empire
The Thin Red Line
Red Cliff
Fort McCoy
Alexander
Underworld: Rise of the Lycans
One Man's Hero
We Were Soldiers
Highlander
Glory


# Study 3 - What if we use other similarity measurement? (not using 'popularity' and 'vote_average')

## Euclidean distance

In [66]:
# Build system 4 distances: Euclidean distance over text vectors
distance4 = euclidean_distances(vectorized)

# Recommendation system 4: text vectors only (Euclidean distance)
def recommendation_system4(movie):

    # Get the index for the input movie title
    index = df_movies_new[df_movies_new['title'] == movie].index[0]

    # For distances, smaller means closer, so we sort ascending
    scores = sorted(list(enumerate(distance4[index])), reverse=True, key=lambda x: x[1])

    # Print the top 10 closest titles (skipping the movie itself)
    for i in scores[1:11]:
        print(df_movies_new.iloc[i[0]].title)


## Manhattan distance

In [67]:
# Build system 5 distances: Manhattan distance over text vectors
distance5 = manhattan_distances(vectorized)

# Recommendation system 5: text vectors only (Manhattan distance)
def recommendation_system5(movie):

    # Get the index for the input movie title
    index = df_movies_new[df_movies_new['title'] == movie].index[0]

    # For distances, smaller means closer, so we sort ascending
    scores = sorted(list(enumerate(distance5[index])), reverse=True, key=lambda x: x[1])

    # Print the top 10 closest titles (skipping the movie itself)
    for i in scores[1:11]:
        print(df_movies_new.iloc[i[0]].title)


### What are the movie recommendations for someone who watched: Spectre using system 4 e 5?

In [68]:
recommendation_system4('Spectre')

Tadpole
The Watcher
The Host
Semi-Pro
The Midnight Meat Train
Rugrats Go Wild
Solomon and Sheba
The Host
Indie Game: The Movie
Flying By


In [69]:
recommendation_system5('Spectre')

The Watcher
The Host
The Midnight Meat Train
The Host
The Matrix Reloaded
Henry & Me
Hart's War
Air Bud
Tadpole
All About the Benjamins


### What are the movie recommendations for someone who watched: Minions using system 4 e 5?

In [70]:
recommendation_system4('Minions')

Tadpole
The Watcher
The Host
The Midnight Meat Train
Semi-Pro
The Host
Solomon and Sheba
Indie Game: The Movie
Flying By
Rugrats Go Wild


In [71]:
recommendation_system5('Minions')

The Watcher
The Host
The Midnight Meat Train
The Host
The Matrix Reloaded
Hart's War
Hercules
Homefront
Henry & Me
Gladiator


### What are the movie recommendations for someone who watched: 300 using system 4 e 5?

In [72]:
recommendation_system4('300')

Tadpole
Rugrats Go Wild
The Host
The Watcher
The Midnight Meat Train
Indie Game: The Movie
Semi-Pro
The Host
Solomon and Sheba
Flying By


In [73]:
recommendation_system5('300')

The Watcher
The Midnight Meat Train
The Host
The Host
Henry & Me
The Matrix Reloaded
Air Bud
Bowling for Columbine
Silent Hill
15 Minutes


# Comparison of the studies

## Movie: Spectre

In [74]:
# Side-by-side comparison for a single title across all 5 systems
import io
from contextlib import redirect_stdout
import pandas as pd

def _capture_recommendations(func, movie, top_n=10):
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        func(movie)
    recs = [line.strip() for line in buffer.getvalue().splitlines() if line.strip()]
    return recs[:top_n]

movie = 'Spectre'

results = pd.DataFrame({
    'System 1 (cosine + popularity + vote_avg)': _capture_recommendations(recommendation_system1, movie),
    'System 2 (cosine + vote_avg)': _capture_recommendations(recommendation_system2, movie),
    'System 3 (cosine, text only)': _capture_recommendations(recommendation_system3, movie),
    'System 4 (Euclidean, text only)': _capture_recommendations(recommendation_system4, movie),
    'System 5 (Manhattan, text only)': _capture_recommendations(recommendation_system5, movie),
})

results


,System 1 (cosine + popularity + vote_avg),System 2 (cosine + vote_avg),"System 3 (cosine, text only)","System 4 (Euclidean, text only)","System 5 (Manhattan, text only)"
0,Quantum of Solace,Quantum of Solace,Quantum of Solace,Tadpole,The Watcher
1,Skyfall,Skyfall,Skyfall,The Watcher,The Host
2,Never Say Never Again,Never Say Never Again,Never Say Never Again,The Host,The Midnight Meat Train
3,From Russia with Love,From Russia with Love,From Russia with Love,Semi-Pro,The Host
4,Octopussy,Octopussy,Octopussy,The Midnight Meat Train,The Matrix Reloaded
5,Safe Haven,Safe Haven,Diamonds Are Forever,Rugrats Go Wild,Henry & Me
6,Diamonds Are Forever,Diamonds Are Forever,Thunderball,Solomon and Sheba,Hart's War
7,Thunderball,Thunderball,Safe Haven,The Host,Air Bud
8,Licence to Kill,Licence to Kill,Licence to Kill,Indie Game: The Movie,Tadpole
9,Dr. No,Dr. No,Dr. No,Flying By,All About the Benjamins


## Movie: Minions

In [75]:
movie = 'Minions'

results = pd.DataFrame({
    'System 1 (cosine + popularity + vote_avg)': _capture_recommendations(recommendation_system1, movie),
    'System 2 (cosine + vote_avg)': _capture_recommendations(recommendation_system2, movie),
    'System 3 (cosine, text only)': _capture_recommendations(recommendation_system3, movie),
    'System 4 (Euclidean, text only)': _capture_recommendations(recommendation_system4, movie),
    'System 5 (Manhattan, text only)': _capture_recommendations(recommendation_system5, movie),
})

results


,System 1 (cosine + popularity + vote_avg),System 2 (cosine + vote_avg),"System 3 (cosine, text only)","System 4 (Euclidean, text only)","System 5 (Manhattan, text only)"
0,Despicable Me 2,Despicable Me 2,Despicable Me 2,Tadpole,The Watcher
1,Penguins of Madagascar,Penguins of Madagascar,The Croods,The Watcher,The Host
2,The Croods,The Croods,Penguins of Madagascar,The Host,The Midnight Meat Train
3,Batman,Batman,Batman,The Midnight Meat Train,The Host
4,Cars 2,Cars 2,Cars 2,Semi-Pro,The Matrix Reloaded
5,Ice Age: Continental Drift,Ice Age: Continental Drift,The Master of Disguise,The Host,Hart's War
6,The Lego Movie,The Lego Movie,Ice Age: Continental Drift,Solomon and Sheba,Hercules
7,Despicable Me,Despicable Me,Valiant,Indie Game: The Movie,Homefront
8,Valiant,The Master of Disguise,Despicable Me,Flying By,Henry & Me
9,The Master of Disguise,Valiant,Free Birds,Rugrats Go Wild,Gladiator


## Movie: 300

In [76]:
movie = '300'

results = pd.DataFrame({
    'System 1 (cosine + popularity + vote_avg)': _capture_recommendations(recommendation_system1, movie),
    'System 2 (cosine + vote_avg)': _capture_recommendations(recommendation_system2, movie),
    'System 3 (cosine, text only)': _capture_recommendations(recommendation_system3, movie),
    'System 4 (Euclidean, text only)': _capture_recommendations(recommendation_system4, movie),
    'System 5 (Manhattan, text only)': _capture_recommendations(recommendation_system5, movie),
})

results


,System 1 (cosine + popularity + vote_avg),System 2 (cosine + vote_avg),"System 3 (cosine, text only)","System 4 (Euclidean, text only)","System 5 (Manhattan, text only)"
0,300: Rise of an Empire,300: Rise of an Empire,300: Rise of an Empire,Tadpole,The Watcher
1,The Thin Red Line,The Thin Red Line,The Thin Red Line,Rugrats Go Wild,The Midnight Meat Train
2,Red Cliff,Red Cliff,Red Cliff,The Host,The Host
3,Fort McCoy,Fort McCoy,Fort McCoy,The Watcher,The Host
4,Alexander,Alexander,Alexander,The Midnight Meat Train,Henry & Me
5,One Man's Hero,One Man's Hero,Underworld: Rise of the Lycans,Indie Game: The Movie,The Matrix Reloaded
6,Underworld: Rise of the Lycans,Underworld: Rise of the Lycans,One Man's Hero,Semi-Pro,Air Bud
7,We Were Soldiers,We Were Soldiers,We Were Soldiers,The Host,Bowling for Columbine
8,Highlander,Highlander,Highlander,Solomon and Sheba,Silent Hill
9,Glory,Glory,Glory,Flying By,15 Minutes


# Conclusion

The results across systems 1, 2, and 3 are nearly identical for all three test movies. Adding `vote_average` or `popularity` to the cosine similarity vectors does not meaningfully change the order of recommendations. This is expected: both numerical features are small scalar values appended to a 5,000-dimensional sparse vector. Their contribution to the dot product used in cosine similarity is negligible compared to the cumulative weight of shared text tokens. A movie with a slightly higher vote score simply cannot overcome the signal from dozens of matched genre, keyword, and cast terms.

Systems 4 and 5 (Euclidean and Manhattan distance) produce results that are inconsistent and clearly incorrect — titles like Tadpole, The Watcher, and The Midnight Meat Train appear as top matches for blockbusters like Spectre and Minions, regardless of the input movie. This happens because Euclidean and Manhattan distances are not well-suited for high-dimensional sparse vectors. In high dimensions, distances between all pairs of vectors tend to converge (the curse of dimensionality), making it hard to distinguish near from far neighbors. Additionally, sorting by the largest distance (as done here) picks the most distant movies in vector space, which, for sparse data, ends up being determined by whichever movies happen to have slightly more non-zero entries in overlapping dimensions, rather than by genuine content similarity. Cosine similarity sidesteps this by measuring the angle between vectors rather than their magnitude, which is far more meaningful when comparing bag-of-words representations.

In [77]:
%reload_ext watermark
%watermark -a "Gui Freire Oliveira"

Author: Gui Freire Oliveira



In [78]:
%watermark -v -m

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-101-generic
Machine     : x86_64
Processor   : x86_64
CPU cores   : 12
Architecture: 64bit



In [79]:
%watermark --iversions

nltk   : 3.9.3
numpy  : 1.26.4
pandas : 2.2.3
sklearn: 1.4.1.post1

